# 🤖 Enterprise AI Copilot Architecture

This notebook implements conversational enterprise AI copilot architecture for the Uber Enterprise Agentic AI Platform.

The objective of this layer is to transform enterprise RAG orchestration into interactive conversational AI systems capable of:

* enterprise question answering
* contextual conversations
* conversational memory
* follow-up query understanding
* contextual semantic retrieval
* grounded AI responses
* operational AI assistance

This notebook covers:

* conversational AI architecture
* conversation memory
* contextual retrieval
* conversational grounding
* enterprise copilots
* multi-turn orchestration
* AI assistant workflows

This notebook represents the transition from single-query RAG pipelines into interactive enterprise AI copilots.


# ⚙️ Environment & Copilot Configuration Initialization

In [0]:
%pip install sentence-transformers

In [0]:
# ==========================================
# Environment & Copilot Configuration
# ==========================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import (
    SentenceTransformer
)

# Enterprise Configuration
CONFIG = {

    "catalog": "spark_catalog",
    "schema": "uber_ai",

    "environment": "dev"
}

# Copilot Configuration
COPILOT_CONFIG = {

    # Embedding Model
    "embedding_model":
        "sentence-transformers/all-MiniLM-L6-v2",

    # Embedding Dimension
    "embedding_dimension":
        384,

    # Top-K Retrieval
    "top_k":
        5,

    # Maximum Conversation Turns
    "max_conversation_turns":
        10
}

print("✅ Copilot Configuration Initialized")
print(COPILOT_CONFIG)

# 🧬 Initialize Enterprise Embedding Model

In [0]:
# ==========================================
# Initialize Enterprise Embedding Model
# ==========================================

# %pip install sentence-transformers

embedding_model = SentenceTransformer(
    
    COPILOT_CONFIG["embedding_model"]
)

print("✅ Enterprise Embedding Model Loaded")

# 🧠 Read Enterprise Vector Registry

In [0]:
# ==========================================
# Read Enterprise Vector Registry
# ==========================================

embedding_vectors_df = spark.table(
    f"{CONFIG['schema']}.embedding_vectors"
)

print("✅ embedding_vectors loaded")

display(
    embedding_vectors_df.limit(5)
)

# 🧠 Enterprise Conversation Memory Initialization

In [0]:
# ==========================================
# Enterprise Conversation Memory Initialization
# ==========================================

# ------------------------------------------
# Initialize Conversation Memory
# ------------------------------------------

conversation_memory = []

print("✅ Enterprise Conversation Memory Initialized")

# ------------------------------------------
# Helper Function
# ------------------------------------------

def add_conversation_turn(

    user_query,
    ai_response
):

    conversation_memory.append({

        "user_query":
            user_query,

        "ai_response":
            ai_response
    })

# ------------------------------------------
# Display Helper
# ------------------------------------------

def display_conversation_memory():

    print("\n")
    print("=" * 100)

    print("ENTERPRISE CONVERSATION MEMORY")

    print("=" * 100)

    for idx, turn in enumerate(conversation_memory):

        print(f"\nTURN {idx+1}")

        print("-" * 50)

        print("USER:")
        print(turn["user_query"])

        print("\nAI:")
        print(turn["ai_response"])

# 🤖 Conversational Semantic Retrieval Pipeline

In [0]:
# ==========================================
# Conversational Semantic Retrieval Pipeline
# ==========================================

# ------------------------------------------
# User Conversational Query
# ------------------------------------------

user_query = (
    
    "What about evening airport demand spikes?"
)

print("✅ User Query:")
print(user_query)

# ------------------------------------------
# Build Conversation Context
# ------------------------------------------

conversation_context = "\n".join([

    f"User: {turn['user_query']}\n"
    f"AI: {turn['ai_response']}"

    for turn in conversation_memory
])

print("✅ Conversation Context Built")

# ------------------------------------------
# Augmented Conversational Query
# ------------------------------------------

augmented_query = f"""

Conversation History:

{conversation_context}

Current User Question:

{user_query}

"""

print("✅ Augmented Conversational Query Constructed")

# ------------------------------------------
# Generate Conversational Embedding
# ------------------------------------------

query_embedding = embedding_model.encode(
    augmented_query
)

print("✅ Conversational Query Embedding Generated")

# ------------------------------------------
# Load Enterprise Vector Registry
# ------------------------------------------

vector_pd = (

    embedding_vectors_df

    .select(
        "chunk_id",
        "chunk_text",
        "semantic_domain",
        "city_name",
        "retrieval_priority",
        "embedding_vector"
    )

    .filter(
        col("city_name") == "Hyderabad"
    )

    .toPandas()
)

print("✅ Enterprise Vector Registry Loaded")

# ------------------------------------------
# Calculate Semantic Similarity
# ------------------------------------------

vector_pd["similarity_score"] = (

    vector_pd["embedding_vector"]

    .apply(

        lambda x:

        cosine_similarity(

            [query_embedding],
            [x]

        )[0][0]
    )
)

print("✅ Conversational Semantic Similarity Calculated")

# ------------------------------------------
# Enterprise Ranking
# ------------------------------------------

vector_pd["final_ranking_score"] = (

    vector_pd["similarity_score"]

    * vector_pd["retrieval_priority"]
)

print("✅ Enterprise Ranking Calculated")

# ------------------------------------------
# Retrieve Top-K Context
# ------------------------------------------

top_context_pd = (

    vector_pd

    .sort_values(
        by="final_ranking_score",
        ascending=False
    )

    .head(
        COPILOT_CONFIG["top_k"]
    )
)

print("✅ Conversational Context Retrieved")

# ------------------------------------------
# Display Retrieved Context
# ------------------------------------------

display(
    spark.createDataFrame(

        top_context_pd[
            [
                "chunk_id",
                "semantic_domain",
                "city_name",
                "similarity_score",
                "final_ranking_score",
                "chunk_text"
            ]
        ]
    )
)

# 🤖 Conversational Grounded Response Orchestration

In [0]:
# ==========================================
# Conversational Grounded Response
# ==========================================

# ------------------------------------------
# Assemble Retrieved Context
# ------------------------------------------

retrieved_context = "\n\n".join(

    top_context_pd["chunk_text"].tolist()
)

print("✅ Retrieved Context Assembled")

# ------------------------------------------
# Construct Conversational Grounding Prompt
# ------------------------------------------

grounded_prompt = f"""

You are an enterprise AI copilot assistant.

Use the conversation history and retrieved
enterprise operational context below to answer
the user's latest question.

If the answer is not available in the context,
say:

'I could not find sufficient enterprise context.'

==================================================
CONVERSATION HISTORY
==================================================

{conversation_context}

==================================================
RETRIEVED ENTERPRISE CONTEXT
==================================================

{retrieved_context}

==================================================
CURRENT USER QUESTION
==================================================

{user_query}

==================================================
ENTERPRISE COPILOT RESPONSE
==================================================

"""

print("✅ Conversational Grounding Prompt Constructed")

# ------------------------------------------
# Simulated Enterprise Copilot Response
# ------------------------------------------

copilot_response = f"""

Based on enterprise operational context,
evening airport demand spikes in Hyderabad
appear strongly correlated with:

1. Increased premium traveler activity

2. Airport congestion during evening hours

3. Surge pricing activation near airport zones

4. Higher ride completion rates for
premium ride categories

Operational retrieval patterns indicate
that evening airport windows consistently
generate elevated premium ride demand.

"""

print("✅ Enterprise Copilot Response Generated")

print("\n")
print("=" * 100)
print("ENTERPRISE COPILOT RESPONSE")
print("=" * 100)

print(copilot_response)

# ------------------------------------------
# Persist Conversation Turn
# ------------------------------------------

add_conversation_turn(

    user_query=user_query,

    ai_response=copilot_response
)

print("✅ Conversation Memory Updated")